# Fine-tuning a Summarization Model

# Introduction
In this chapter, we will explore how to use a Transformer model to convert long documents into short, concise summaries — a task known as Text Summarization.

This is one of the most challenging tasks in NLP because it requires:
The ability to understand long passages of text
The ability to retain key ideas and express them in coherent language
But when it works well, it significantly accelerates business workflows — experts no longer need to read entire lengthy documents.
One particularly exciting aspect of this chapter: we will build a bilingual model that can handle both English and Spanish.

By the end, our model will be able to summarize customer reviews like this:

"I loved The dash diet weight loss Solution. Never hungry. I would recommend this diet." → Summary: "Easy to follow!!!!"



# Building the Multilingual Corpus
Dataset Selection

We will use the mteb/amazon_reviews_multi dataset, which is a publicly available mirror of the original Multilingual Amazon Reviews Corpus. It contains Amazon product reviews in six languages. Each review comes with a short title — those titles will serve as our target summaries.

Why this dataset? The original amazon_reviews_multi dataset was taken down by its data providers and is no longer accessible. The mteb/amazon_reviews_multi dataset is a publicly available version with the exact same structure and fields.
Let's load the English and Spanish datasets:


In [ ]:
'''
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mexwell/amazon-reviews-multi",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())
'''

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_dataset = pd.read_csv("/kaggle/input/datasets/mexwell/amazon-reviews-multi/train.csv")
test_dataset = pd.read_csv("/kaggle/input/datasets/mexwell/amazon-reviews-multi/test.csv")
val_dataset = pd.read_csv("/kaggle/input/datasets/mexwell/amazon-reviews-multi/validation.csv")


In [ ]:
train_dataset

In [ ]:
test_dataset

In [ ]:
val_dataset

## Filter only English reviews

In [ ]:
english_dataset = train_dataset[train_dataset["language"] == "en"]

# Display first few rows
print(english_dataset.head())

# Check the shape of the filtered dataset
print("Shape of English Dataset:", english_dataset.shape)

In [ ]:
english_dataset

## Filter only Spanish reviews

In [ ]:
spanish_dataset = train_dataset[train_dataset["language"] == "es"]

# Display first few rows
print(spanish_dataset.head())

# Check the shape of the filtered dataset
print("Shape of spanish dataset:", spanish_dataset.shape)

In [ ]:
spanish_dataset

## Exploring the Data
Let's write a small function to look at a few random samples from the english_dataset:


In [ ]:
title_review = english_dataset[['review_title' ,'review_body']]

In [ ]:
title_review 

## Analyzing Product Categories


Training on 200,000 reviews would take a very long time on a single GPU, so we'll focus on a specific product category.

In [ ]:
# Show counts for top 20 product categories
english_dataset["product_category"].value_counts()[:20]

Amazon started as a bookstore, so let's focus on the book and digital_ebook_purchase categories!
## Filtering for Books


In [ ]:
def filter_books(example):
    return(
        example["product_category"] == "book"
        or example["product_category"] == "digital_ebook_purchase"
    )

Before applying the filter, let's reset the dataset format back to "arrow":


Now apply the filter and preview some samples:


In [ ]:
spanish_books = spanish_dataset[spanish_dataset.apply(filter_books, axis=1)]
english_books = english_dataset[english_dataset.apply(filter_books, axis=1)]

english_books

In [ ]:
english_books.keys()

In [ ]:
spanish_books

In [ ]:
spanish_books.keys()

Notice that the reviews aren't exclusively about books — there are calendars and digital tools like OneNote. But the domain is still good enough to train a summarization model.


## Building the Bilingual Dataset

Now let's combine the English and Spanish reviews into a single DatasetDict. We'll use the concatenate_datasets() function from the datasets library to join them side by side:

In [ ]:
from datasets import Dataset, concatenate_datasets, DatasetDict

# pandas DataFrame থেকে HuggingFace Dataset বানানো
spanish_books_ds = Dataset.from_pandas(spanish_books)
english_books_ds = Dataset.from_pandas(english_books)

books_dataset = DatasetDict()

# ধরুন আপনার split আছে (train/test ইত্যাদি)
for split in ["train"]:  # বা english_books.keys() যদি dict হয়
    books_dataset[split] = concatenate_datasets(
        [english_books_ds, spanish_books_ds]
    )
    books_dataset[split] = books_dataset[split].shuffle(seed=42)

# Peek few examples
print(books_dataset)


In [ ]:
# lets books_dataset["train"] is your Dataset
sample_rows = books_dataset["train"].shuffle(seed=123).select(range(5)) 

# print only 2 columns
for row in sample_rows:
    print("Title:", row["review_title"])
    print("Body :", row["review_body"])
    print("-" * 40)


We can see a mix of English and Spanish reviews.


## Checking Word Distribution & Filtering Short Titles


Before preparing the training corpus, it's important to look at the distribution of words in the reviews and titles. This is especially critical in summarization tasks, because if the reference summaries are very short (e.g., 1–2 words), the model will learn to produce only 1–2 word outputs.

To avoid this, let's filter out very short titles so the model learns to generate more informative summaries:


In [ ]:
books_dataset = books_dataset.filter(lambda x : len(x["review_title"].split()) > 2)

# Models for Text Summarization
Think of Summarization Like Translation

Text summarization is similar to machine translation — we are "translating" a review into its condensed version. This is why most Transformer summarization models use an encoder-decoder architecture.

Here is a summary of popular pretrained models:
Model -- Description -- Multilingual?
- GPT-2 -- Auto-regressive language model. Can generate summaries if you append "TL;DR" to the input. ❌
- PEGASUS -- Uses masked sentence prediction as pretraining objective. Scores highly on popular summarization benchmarks.-- ❌
- T5 -- Completes all NLP tasks in a text-to-text framework. Input format for summarization: summarize: ARTICLE.-- ❌
- mT5 -- Multilingual version of T5. Pretrained on the multilingual Common Crawl corpus (mC4) in 101 languages. -- ✅
- BART -- Has both encoder and decoder. Combines BERT and GPT-2 pretraining by learning to reconstruct corrupted input.-- ❌
- mBART-50 -- Multilingual version of BART, pretrained on 50 languages.-- ✅

#### Why mT5?
We'll use mT5. In T5, every NLP task is handled with a prompt prefix (e.g., summarize:). mT5 doesn't use prefixes, but it is just as versatile as T5 and has the added benefit of being multilingual.

Try it out! After finishing this chapter, compare mT5 vs mBART. As a bonus, try fine-tuning T5 on English-only reviews — just remember to prepend summarize: to the input during preprocessing.


# Data Preprocessing

## Loading the Tokenizer
We'll use the mt5-small checkpoint so we can fine-tune the model in a reasonable amount of time:

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


#### Tip:
When starting an NLP project, always begin with a "small" model on a subset of your data. This allows you to debug and iterate quickly. Once you're satisfied with the results, simply swap the model checkpoint to scale up!

Let's test the tokenizer on a small example:


In [ ]:
inputs = tokenizer("I love reading the hunger games")
inputs

Let's decode the input IDs to understand what tokenizer is being used:


In [ ]:
tokenizer.convert_ids_to_tokens(inputs.input_ids)

The special Unicode character ▁ and the end-of-sequence token </s> tell us this is a SentencePiece tokenizer, which uses the Unigram segmentation algorithm. Unigram is particularly well-suited for multilingual corpora because it handles accents, punctuation, and whitespace-free languages (like Japanese) in an agnostic way.

## The Preprocess Function

There is a subtle point in summarization: our labels are also text, so they could exceed the model's maximum context size. Therefore, we must apply truncation to both the reviews and the titles:


In [ ]:
max_input_length = 512
max_target_length = 30

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        examples["review_title"],
        max_length=max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


Here, max_input_length = 512 is set for reviews and max_target_length = 30 for titles — because reviews are typically much longer than titles.

Now let's tokenize the entire corpus using Dataset.map():


In [ ]:
tokenized_datasets = books_dataset.map(preprocess_function,batched=True)

#### Tip:
Using batched=True processes data in batches of 1,000 examples by default, and takes advantage of multithreading in Transformers' fast tokenizers.


# ROUGE Score: The Summarization Metric


## Why ROUGE?
Measuring text generation is not as straightforward as classification. A single review can have many valid summaries, and doing an exact match between generated and reference summaries is not meaningful.

The most widely used metric for summarization is the ROUGE score (Recall-Oriented Understudy for Gisting Evaluation). The core idea is to compare a generated summary against a human-written reference summary.

## Computing Precision and Recall
Suppose we have:


In [ ]:
generated_summary = "I absolutely loved reading the hunger games"
reference_summary = "I loved reading the hunger games"

Recall measures how much of the reference summary is captured in the generated summary:

Recall = (overlapping words) / (total words in reference)
      
       = 6 / 6 = 1.0  ← perfect recall


But what if our generated summary was "I really really loved reading the Hunger Games all night"? Recall would still be perfect, but the summary is verbose. This is why we also need Precision:

Precision = (overlapping words) / (total words in generated summary)

- Verbose summary:  6/10 = 0.6  ← much worse
- Concise summary:  6/7  = 0.86 ← much better


## Installing and Using ROUGE


#### Code
!pip install rouge_score

import evaluate

rouge_score = evaluate.load("rouge")

scores = rouge_score.compute(

    predictions=[generated_summary],
    
    references=[reference_summary]

)

scores

#### Output:
{'rouge1': AggregateScore(low=Score(precision=0.86, recall=1.0, fmeasure=0.92), ...),

 'rouge2': AggregateScore(low=Score(precision=0.67, recall=0.8, fmeasure=0.73), ...),
 
 'rougeL': AggregateScore(low=Score(precision=0.86, recall=1.0, fmeasure=0.92), ...),
 
 'rougeLsum': AggregateScore(...)}


#### There's a lot of information here. Let's break it down:

The datasets library computes confidence intervals for precision, recall, and F1-score — shown as low, mid, and high.
- rouge1 — overlap of individual words (unigrams)
- rouge2 — overlap of word pairs (bigrams)
- rougeL — longest matching sequence of words (longest common substring)
- rougeLsum — rougeL computed over the full summary

Let's inspect the mid score:

scores["rouge1"].mid

#### Output:
Score(precision=0.86, recall=1.0, fmeasure=0.92)


The numbers match our manual calculation perfectly.


# Building a Strong Baseline

## The Lead-3 Baseline
A common baseline for text summarization is to take the first three sentences of an article — this is called the lead-3 baseline.

If we naively split on full stops, abbreviations like "U.S." or "U.N." would cause problems. So we'll use the nltk library instead:


In [ ]:
pip install nltk

In [ ]:
import nltk
nltk.download("punkt")

In [ ]:
from nltk.tokenize import sent_tokenize

def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])

print(three_sentence_summary(books_dataset["train"][1]["review_body"]))

Now let's compute ROUGE scores on the validation set:


In [ ]:
def evaluate_baseline(dataset,metric):
    summaries = [three_sentence_summaryt(text) for text in dataset["review_body"]]
    return metric.compute(predictions = summaries , references=dataset["review_title"])

In [ ]:
import pandas as pd

score = evaluate_baseline(books_dataset["validation"],rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, round(score[rn].mid.fmeasure * 100, 2)) for rn in rouge_names)
rouge_dict


#### Output:
{'rouge1': 16.74, 'rouge2': 8.83, 'rougeL': 15.6, 'rougeLsum': 15.96}



The rouge2 score is quite low — because review titles are typically very concise, while the lead-3 baseline is far more verbose.

#  Fine-Tuning mT5 with the Trainer API


## Loading the Model


In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)


For sequence-to-sequence tasks, we keep all of the network's weights. Unlike text classification where we had to swap out the head, there's no need for that here.
## Logging in to Hugging Face Hub


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Setting Training Arguments


In [ ]:
from transformers import Seq2SeqTrainingArguments

batch_size = 8
num_train_epoch = 8
logging_steps = len(tokenized_datasets["train"])//batch_size
model_name = model_checkpoint.split("\n")[-1]

args = Seq2SeqTraningArguments(
    output_dir=f"{model_name}-finetuned-amazon-en-es",
    evaluation_strategy = "epoch",
    learning_rate =5.6e-5,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps = logging_steps,
    push_to_hub = True,

)

### Key points:
- predict_with_generate=True makes the model generate actual summaries during evaluation (instead of just computing loss) so we can calculate real ROUGE scores each epoch.
- save_total_limit=3 keeps only the 3 most recent checkpoints — even the "small" version of mT5 takes about 1 GB of disk space.
- push_to_hub=True automatically uploads the model to the Hub when training is done.

## The compute_metrics Function


We need a compute_metrics() function to calculate ROUGE scores during training. This involves decoding the predicted token IDs and reference labels back to text:


In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode generated summaries into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels (we can't decode them)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode reference summaries into text
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE expects a newline after each sentence
    decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]

    # Compute ROUGE scores
    result = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # Extract the median scores
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return {k: round(v, 4) for k, v in result.items()}


## The Data Collator
mT5 is an encoder-decoder Transformer, so there's a subtle detail when preparing batches: during decoding, the labels must be shifted one step to the right. This ensures the decoder only sees previous ground-truth labels, not the current or future ones.

Transformers' DataCollatorForSeq2Seq handles this automatically:

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer,model=model)


We also need to remove the string columns, since the collator can't pad them:

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(
    books_dataset["train"].column_names
)

Let's test the collator on a small batch:


In [ ]:
features = [tokenized_datasets["train"][i] for i in range(2)]
data_collator(features)

Things to notice:
- The first example is shorter than the second, so [PAD] tokens (ID 0) are appended to its input_ids and attention_mask.
- The labels are padded with -100 so the loss function ignores them.
- The decoder_input_ids have the labels shifted one step to the right, with [PAD] at the start.

## Creating the Trainer and Starting Training

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    data_collator = data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

During training, you will see the training loss decrease and the ROUGE scores increase with each epoch.


In [ ]:
## Final Evaluation
trainer.evaluate()


In [ ]:
#Pushing to the Hub
trainer.push_to_hub(commit_message="Training complete", tags="summarization")
